# フィルム成長パラメタの最適化

参考文献： Ohkubo et al., Materials Today Physics 16 (2021) 100296


## 手法

金属有機分子線エピタキシー（MO-MBE）を超高真空で行ない、SrTiO₃(002)基板上にITiN(002)を成長させる。

## 調整する4つの成長パラメータ（説明変数）

x₁, "Growth Temperature (°C)": 成長（基板）温度 Ts（約 550–900 °C）

x₂, "TDMAT Pressure (Torr)": Ti 源（TDMAT）の供給圧（前段フォーライン圧で表記、~1.7–6.3 Torr 相当）

x₃, "N2 Gas Flow (sccm)": N₂プラズマのRF電力（~300–500 W程度の範囲で）

x₄, "N2 Plasma RF Power (W)": N₂ 流量（~0.5–4.5 sccm 付近）
これらを実験の制御分解能に合わせた格子上で探索します。

## 目的変数 (Normalized XRD Intensity (Inorm = ITiN(002)/ISTO(002)d)) の定義と目的

ITiN(002)：TiN(002)回折ピーク強度

ISTO(002)：SrTiO₃(002)基板ピーク強度（測定毎の幾何・装置要因のばらつき基準）

d：膜厚（表面段差計で測定）

Inorm = ITiN(002) / [ ISTO(002) · d ] を目的関数として最大化＝「厚さと測定条件の影響を除いた、相対的な(002)配向結晶性の指標」を評価する。

## 結晶性の高い試料を作成する目的

参考文献では、最も良い条件で作成したMgO(001)上の TiN 膜は Tc(onset) ≈ 5.25 K、Tc(zero) ≈ 5.07 K と高い超伝導特性を示した。




In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def get_data(data_name="epitaxy-film-growth"):
    feat_cols = ["Growth Temperature (°C)", "TDMAT Pressure (Torr)", 
                 "N2 Gas Flow (sccm)", "N2 Plasma RF Power (W)"]
    target_col = "log10 Normalized XRD Intensity (Inorm = ITiN(002)/ISTO(002)d)"
    
    df_all = pd.read_csv("../data_calculated/Epitaxial-thin-filmgrowth.csv")

    return df_all, feat_cols, target_col

# epitaxy-filmの成長パラメタと仮想的なXRD intensityを取得する。
g_df_all, g_feat_cols, g_target_col = get_data()

# 目的変数の最大値を求めておく。
g_target_max = g_df_all[g_target_col].max()
print(f"Maximum of '{g_target_col}' = {g_target_max}")

g_df_all

In [ ]:
import pickle

def load_model(filename = "../data_calculated/Epitaxial-thin-filmgrowth.pkl"):
    """
    保存済みの学習済みパイプラインを読み込み、スケーラとPCAオブジェクトを返す関数。

    Parameters
    ----------
    filename : str, optional
        pickle 形式で保存された学習済みパイプラインのファイルパス。
        デフォルトは "model/best_pipe.pkl"。

    Returns
    -------
    scaler :
        読み込んだパイプライン内の "scaler" ステップ（例: StandardScaler など）。
    pca :
        読み込んだパイプライン内の "pca" ステップ（例: PCA など）。

    Notes
    -----
    本関数は、`sklearn.pipeline.Pipeline` を pickle で保存したファイルを前提としており、
    `named_steps["scaler"]` および `named_steps["pca"]` が存在することを期待している。
    """
    with open(filename, "rb") as f:
        best_pipe = pickle.load(f)
    scaler = best_pipe.named_steps["scaler"]
    pca = best_pipe.named_steps["pca"]
    return scaler,pca

g_X_scaler,g_pca = load_model()
g_X_scaler, g_pca

In [ ]:
def plot_pca_axis(pca, feat_cols):
    loadings = pd.DataFrame(pca.components_.T, 
                            columns=['PC1', 'PC2'], 
                            index=feat_cols)
    print(loadings)
    
    plt.figure(figsize=(6,6))
    for i, feature in enumerate(feat_cols):
        plt.arrow(0, 0, loadings.PC1[i], loadings.PC2[i],
                  color='r', alpha=0.7, head_width=0.02)
        plt.text(loadings.PC1[i]*1.1, loadings.PC2[i]*1.1, feature, color='r')
    plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.title("PCA Loadings (Feature Contributions)")
    plt.grid(True)
    plt.axhline(0, color='gray', lw=0.5)
    plt.axvline(0, color='gray', lw=0.5)
    plt.show()

plot_pca_axis(g_pca, g_feat_cols)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_all_data(df_all, scaler, pca, feat_cols, target_col):
    """
    全データを PCA 2 次元空間に写像し、目的変数で着色した散布図を描画する。

    本関数は、
    1. 特徴量列 `feat_cols` を取り出してスケーリング
    2. PCA により 2 次元（第1主成分・第2主成分）へ次元削減
    3. `target_col` の値で色付けした散布図を描画
    を行う。

    Parameters
    ----------
    df_all : pandas.DataFrame
        解析対象となる全サンプルを含む DataFrame。
        少なくとも `feat_cols` と `target_col` の列を含んでいる必要がある。
    scaler :
        学習済みのスケーラオブジェクト（例: `sklearn.preprocessing.StandardScaler`）。
        `scaler.transform(X_all)` が呼び出せることを前提とする。
    pca :
        学習済みの PCA オブジェクト（例: `sklearn.decomposition.PCA`）。
        `pca.transform(X_all_scaled)` が呼び出せ、少なくとも 2 成分を持つことを前提とする。
    feat_cols : list of str
        特徴量として用いる列名のリスト。
    target_col : str
        散布図の色として使用する目的変数の列名。

    Notes
    -----
    - 色は `target_col` に対応する値（例: 観測された正規化 log10 XRD 強度）で表現される。
    - `pca.explained_variance_ratio_[0]` および `[1]` を用いて、PC1/PC2 の寄与率を軸ラベルに表示する。
    - 図はその場で `plt.show()` により表示される（戻り値はない）。
    """

    # 仮想的な観測値
    X_all = df_all[feat_cols].values 
    y_log_obs = df_all[target_col].values
    
    X_all_scaled = scaler.transform(X_all)
    X_all_pca = pca.transform(X_all_scaled)
    
    # === 4. 散布図（色 = 予測された XRD Intensity） ===
    plt.figure(figsize=(6,4))
    sc = plt.scatter(
        X_all_pca[:,0], X_all_pca[:,1],
        c=y_log_obs, cmap="viridis", s=10, edgecolors="none"
    )
    plt.colorbar(sc, label="Observed Normalized log10 XRD Intensity")
    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
    plt.title("Observed XRD Intensity on PCA(2D) Space")
    plt.grid(True)
    plt.show()

plot_all_data(g_df_all,  g_X_scaler, g_pca, g_feat_cols, g_target_col)